# Importing libraries and data 

_End-to-end retail PD scorecard: data prep -> stratified split -> WoE/IV feature engineering -> logistic regression champion -> validation (AUC/Gini/KS + bootstrap CIs) -> PDO scaling -> calibration -> MLflow artifact lock._

In [0]:
# =============================================================================
# Environment setup: install pipeline dependencies.
#   statsmodels -> unregularised logistic regression (the regulatory champion)
#   kagglehub   -> programmatic pull of the public dataset
#   optbinning  -> WoE/IV optimal binning (outlier + missing + monotonicity handling)
# =============================================================================
!pip install statsmodels
!pip install kagglehub
!pip install optbinning

In [0]:
# =============================================================================
# Core imports.
#   OptimalBinning / BinningProcess -> WoE transform + Information Value
#   variance_inflation_factor       -> multicollinearity diagnostic on WoE features
#   statsmodels (sm)                -> GLM/logit with p-values for a documentable model
#   chi2_contingency                -> significance test for informative missingness
#   combinations                    -> pairwise Cramer's V across categorical vars
# =============================================================================
import pandas as pd
import itertools
import numpy as np
from optbinning import OptimalBinning,BinningProcess
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
import statsmodels.api as sm
import sklearn as sk
from sklearn.metrics import roc_auc_score
import kagglehub as kh
from sklearn.model_selection import train_test_split
from scipy.stats import chi2,chi2_contingency
from itertools import combinations
import os
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

In [0]:
# =============================================================================
# Pull the public Kaggle credit-risk dataset and load the untouched file.
# raw_df is the single source of truth BEFORE any cleaning/transformation.
# =============================================================================
#Backup Path of the file from DataBricks Volumes
#raw_df = pd.read_csv("/Volumes/workspace/default/finance_data/credit_risk_dataset.csv")


# Download latest version of the 
path = kh.dataset_download("laotse/credit-risk-dataset")
print(os.listdir(path))
full_path = os.path.join(path,'credit_risk_dataset.csv')
print(full_path)

In [0]:
#Loaded the raw dataset
raw_df = pd.read_csv(full_path)

# Data Legend 

**Feature Name**:	Description
- **person_age**:	Age
- **person_income**: 	Annual Income
- **person_home_ownership**:	Home ownership
- **person_emp_length**:	Employment length (in years)
- **loan_intent**:	Loan intent
- **loan_grade**:	Loan grade
- **loan_amnt**:	Loan amount
- **loan_int_rate**:	Interest rate
- **loan_status	Loan**: status (0 is non default 1 is default)
- **loan_percent_income**:	Current Loan as a percent of Income
- **cb_person_default_on_file**:	Historical default
- **cb_preson_cred_hist_length**:	Credit history length


# Phase 1: Data Preperation 

Goal of this phase: establish data quality, surface impossible values and informative missingness, and decide treatments.

## Data Quality Sanity Check 

In [0]:
# Eyeballing first rows: confirming the file loaded and columns match the data legend.
raw_df.head(3)

In [0]:
# person_emp_length and loan_int_rate are the two columns carrying missing values.

raw_df.info()

In [0]:
# Target prevalence 
### Profile of customers ( Defaults )
print(f'The Number of Loans:{len(raw_df['loan_status'])}')
print(f'The Number of Goods:{raw_df['loan_status'].value_counts()[0]}')
print(f'The Number of Bads:{raw_df['loan_status'].value_counts()[1]}')
### ~28% customers are default in the data 
default_rate = round(raw_df['loan_status'].mean(),2)
print('\n')
print(f'The Default Rate is :{default_rate}%')

In [0]:
# Integrity check on the complete (NON-NULL) COLUMNS ONLY: confirms every observed
# value is accounted for, i.e. no stray/hidden categories outside the two null fields.
## Checking for unseen values/bad data apart from null in columns that have full length
columns_list = raw_df.columns.tolist()
columns_list.remove('person_emp_length')
columns_list.remove('loan_int_rate')
for c in columns_list:
    sum_list = []
    for i in raw_df[c].unique():
        sum_list.append(raw_df[c].value_counts()[i])
    print(f'The column:{c} has all values accounted for : {np.sum(sum_list) == len(raw_df[c])}')

In [0]:
#Documenting the missingness assumptions:

# Inspect rows where emp_length is null but income exists -> treat as 'data not supplied' rather than 'unemployed'. 
## Similarly for Loan amnt - assuming data was unavailable
raw_df[raw_df['person_emp_length'].isna()].head(3)

In [0]:
# Range scan (min/max) per column to surface impossible values.
# person_age shows an implausible maximum -> data corruption.

for i in raw_df.columns.tolist():
    print(f'Max value for {i} : {raw_df[i].max()} , Min : {raw_df[i].min()}')

In [0]:
# Flag biologically impossible ages (>=122, the verified human longevity ceiling).
# These are corrupted records; WoE binning will isolate them rather than hard-deleting.

raw_df[raw_df['person_age'] >= 122]['person_age'].value_counts()

In [0]:
# Drill into the most extreme age (144) to confirm it is a data-entry artefact.
raw_df[raw_df['person_age']  == 144 ]

## Exploratory Data Analysis 

In [0]:
# Five-number summary. Surfaces person_income skew and the loan_percent_income
# (FOIR / exposure) tail that will need treatment.
## person_income needs further analysis
## loan_percent_income needs to be capped - Reason: people with high debt are unlikely to get loans and hence should not have been approved for a loan in the first place
raw_df.describe()

In [0]:
# Income summary as integers for readability.
raw_df.describe()['person_income'].astype(int)

In [0]:
# Re-confirm target counts.
raw_df['loan_status'].value_counts()

In [0]:
# Isolate numeric columns for correlation / skew / distribution analysis.
raw_df_numeric = raw_df.select_dtypes(include=['number'],exclude=['object'])
raw_df_numeric.info()

In [0]:
# Income histogram -> heavy right skew, motivating WoE binning over raw linear use.
raw_df_numeric['person_income'].hist()

In [0]:
# Size the income tail: compare 2.5th pct, mean, and 97.5th pct.
print(np.percentile(raw_df_numeric['person_income'],2.5))
print(raw_df_numeric['person_income'].mean())
print(np.percentile(raw_df_numeric['person_income'],97.5))

In [0]:
# Skewness per numeric column. Income is extreme -> confirms WoE/binning instead of feeding raw values into the linear logit.

raw_df_numeric.skew()

In [0]:
# Top 10 incomes -> The tail shows that the top 10 incomes are genuine.
raw_df['person_income'].sort_values(ascending=False).head(10)

In [0]:
# Pearson correlation (numeric features).

#Result:
# person_age vs credit-history-length are highly correlated -> keep only the higher-IV one to avoid multicollinearity that would destabilise coefficients.
# Persons credit hist length is highly correlated with person_age - will keep the one with higher iv
# loan amnt and loan percent income show high correlation HOWEVER both have business meaning - loan amnt gives the magnitude of current loan and loan_percent_income gives the overall exposure (FOIR) of customer - will investigate further with IV 
raw_df_numeric.corr()

In [0]:
# Values for person_income is showing high skewness - values will be clipped at the 99 percentile level
# cb_person_cred_hist_length and person_age are highly correlated - will drop value with lower iv 
# loan_percent_income needs to be capped - Reason: people with high debt are unlikely to get loans and hence should not have been approved for a loan in the first place

In [0]:
# Categorical subset for association testing.
# Using cramers v to check correlation between the cat variables 
raw_df_cat = raw_df.select_dtypes(include = ['object'])
raw_df_cat

In [0]:
# Cramer's V: normalises the chi-square statistic into a 0-1 association measure
# between two categorical variables. This is an EFFECT SIZE, not just significance.
# On large credit portfolios p-values are almost always significant, so Cramer's V
# (not the p-value) is the operationally meaningful redundancy metric.
def cramers_v(x,y):
    ### making a confusion matrix
    table = pd.crosstab(x,y)
    ### Calculating chi2 statistic
    chi2 = chi2_contingency(table)[0]
    ### calculating n , r and c
    n = table.sum().sum()
    r , c = table.shape
    ## calculating cramers v
    return np.sqrt((chi2)/(n*(min(r,c)-1)))


In [0]:
# Build the pairwise Cramer's V matrix across all categorical features.
cols = raw_df_cat.columns
cramers_matrix = pd.DataFrame(np.eye(len(cols)),index=cols,columns=cols)

for col1,col2 in combinations(cols,2):
    v = cramers_v(raw_df_cat[col1],raw_df_cat[col2])
    cramers_matrix.loc[col1,col2] = v
    cramers_matrix.loc[col2,col1] = v

In [0]:
# Inspect the association matrix.
cramers_matrix

In [0]:
#Cramers value between cb_person_default_on_file and loan_grade is high 

### Final Actions Required
- person_emp_length and loan_int_rate have null values - will asign WoE from Binning
- Values for person_income is showing high skewness - values will be binned by woe 
- cb_person_cred_hist_length and person_age are highly correlated - will drop value with lower iv 
- Cramers value between cb_person_default_on_file and loan_grade is high. - will drop value with lower iv

#Phase 2: Sampling - The Critical Split 

_Here the data will be split based on an Out of Sample Validation. A Training set and a Test set will be made , both Stratified. Test size will be 20%_

In [0]:
# Random Stratified Holdout chosen because the data has NO time dimension
# (no origination dates) -> a time-based out-of-time (OOT) split is not possible here.
#Given the data is non-time based. Will be using Random Stratified Holdout.

In [0]:
# Separate target (Y) from features (X) BEFORE the split.
# Nothing has been transformed yet -> preserves leakage discipline.
Y = raw_df['loan_status']
X = raw_df.drop(columns='loan_status')

In [0]:
# 80/20 STRATIFIED split.
# Setting Random State to 65 for reproducibility
X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size = 0.2 ,stratify = Y ,random_state = 65)

In [0]:
# Confirm shapes line up.
X_train.shape,Y_train.shape,X_test.shape,Y_test.shape

In [0]:
# Verify stratification held: train and test bad rates should both match ~22%.
# Checking for correct stratification 
print(Y_train.value_counts(normalize=True).to_frame()) 

print(Y_test.value_counts(normalize=True).to_frame())

#Phase 3: Feature Engineering and Selection

_Treatments are decided on TRAIN only: median-impute non-informative missings, give informative missings their own WoE bin, drop redundant/low-IV/model-output features (loan_grade, age, credit-history-length)._

## Missing Value Treatment 

In [0]:
null_test_df = raw_df[['person_emp_length','loan_int_rate','loan_status']]
null_test_df_train_idx = null_test_df.iloc[X_train.index]
null_test_df_train_idx

In [0]:
# Default rate by null-vs-non-null for emp_length.
# A >3pp gap in the default rate => the missingness is INFORMATIVE -> we give it its own WoE bin (do not impute).

null_test_df_train_idx.groupby(null_test_df_train_idx['person_emp_length'].isna())['loan_status'].mean()

In [0]:
# Same test for loan_int_rate. Gap ~= 0 => missingness is NON-informative
# -> median-impute. The median is learned on TRAIN and reused on test.
## Very less difference between two groups - will impute median for null values
print(null_test_df_train_idx.groupby(null_test_df_train_idx['loan_int_rate'].isna())['loan_status'].mean())
median_value = X_train['loan_int_rate'].median()
print(median_value)
X_train['loan_int_rate'] = X_train['loan_int_rate'].fillna(median_value)
X_train['loan_int_rate'].isna().sum() == 0

In [0]:
# Chi-square on emp_length missingness vs default.
# A significant p-value confirms the >3pp gap is not noise -> justifies a Missing WoE bin.
## Chi2 is showing significant statistical different for emp length null values - will use WOE BIN here in later phase 
contingency = pd.crosstab(null_test_df_train_idx['person_emp_length'].isna(),null_test_df_train_idx['loan_status'])
chi2 = chi2_contingency(contingency)
round(chi2.pvalue.astype(float),4)

## Outlier Treatment

### Comments
- Outliers will be treated by WoE Binning 

## Catagorical Correlation 

In [0]:

## Loan Grade is an external risk model score used to predict risk. Hence will be dropped as it may produce a dominant coefficient. In addition our model may just act as a wrapper for this and hence needs to be dropped.

optb = OptimalBinning(name='loan_grade',dtype='categorical')
optb.fit(X_train['loan_grade'],Y_train)
optb.binning_table.build()

In [0]:
# cb_person_default_on_file (historical default flag): strong, economically sensible IV -> RETAIN.
## Good IV 
optb2 = OptimalBinning(name='cb_person_default_on_file',dtype='categorical')
optb2.fit(X_train['cb_person_default_on_file'],Y_train)
optb2.binning_table.build()

In [0]:
# Drop loan_grade from TRAIN per the rationale above.
del X_train['loan_grade']

### Comment
- Dropping Loan_Grade

## Numerical Correlation 

In [0]:
# IV for person_age = 0.0122
opt4 = OptimalBinning(name='person_age',dtype='numerical')
opt4.fit(X_train['person_age'],Y_train)
opt4.binning_table.build()

In [0]:
# credit-history-length: low IV and correlated with age -> DROP.
# Low IV - Dropping 
optb3 = OptimalBinning(name='cb_person_cred_hist_length',dtype='numerical')
optb3.fit(X_train['cb_person_cred_hist_length'],Y_train)
optb3.binning_table.build()

In [0]:
del X_train['cb_person_cred_hist_length']

In [0]:
# person_age: low IV (plus the corrupted tail) -> DROP. Keep the cleaner correlate only.
# Low IV - Dropping 

optb4 = OptimalBinning(name='person_age',dtype='numerical')
optb4.fit(X_train['person_age'],Y_train)
optb4.binning_table.build()

In [0]:
del X_train['person_age']

In [0]:
# loan amnt and loan percent income both have business meaning - loan amnt gives the magnitude of current loan and loan_percent_income gives the overall exposure (FOIR) of customer - will investigate further with IV 

In [0]:
# loan_amnt shows a PEAK-shaped risk profile: thin-file small loans are risky, risk
# eases in the middle, then rises again for very large loans (higher repayment burden).
# Assuming that as the limit of the current loan approved is higher the customer is less risky - Setting monotonic_trend as ascending
 


optb5 = OptimalBinning(name='loan_amnt',dtype = 'numerical',monotonic_trend = 'ascending')
optb5.fit(X_train['loan_amnt'],Y_train)
optb5.binning_table.build()

In [0]:
# Variable shows iv of 0.95 indicating strong univariate predictive power. the top 2 bins capture ~38% of the bads as compared to <20% in lower bins. The Variable is retained but its dominance is acknowledged. It will likely carry the largest scorecard contributions , with over variables offering incremental lift.
optb6 = OptimalBinning(name='loan_percent_income',dtype = 'numerical')
optb6.fit(X_train['loan_percent_income'],Y_train)
optb6.binning_table.build()

### Final Notes
- For Missing Values : Imputed Median for loan_int_rate as the difference between null and non null on defaults was close to 0. 
- For person_emp_length will be binned using WoE as Null values show statistical significance wrt default

- For Outliers : WoE will be used.

- For Catagorical Correlation: Loan Grade is dropped.
 cb_person_default_on_file is kept due to good iv

- For Numerical Correlation: Both cb_person_cred_hist_length and loan_percent_income are kept. 
 person_age is dropped due to low IV 

# Phase 4: Binning and WoE Transformation (Train Data)

_WoE replaces each raw value with ln(odds) of the bin, simultaneously handling outliers, missings and non-linearity, and linearising the relationship for logistic regression. Bins are fit on TRAIN and applied unchanged to TEST._

## Optimal Binning 

In [0]:
# Confirm the surviving feature set on TRAIN before fitting the binning process.
X_train.head()

In [0]:
# Fit the BinningProcess on TRAIN ONLY.
binning_fit_params = {'loan_amnt':{'monotonic_trend':'peak'}}

optb_f = BinningProcess(variable_names = X_train.columns.tolist(),binning_fit_params = binning_fit_params,min_bin_size=0.05)
binned = optb_f.fit(X_train,Y_train)

binned.summary()

In [0]:
# Checked for each variable using : 
#binned.get_binned_variable('cb_person_default_on_file').binning_table.build()

### Comment
- all variables showing monotinicity

## WoE Transformation on Train and Test Data 

In [0]:
# Transform TRAIN features to WoE using the fitted bins.
X_train_t = binned.transform(X_train,metric='woe')
X_train_t.head()

In [0]:
# Transform TEST with the SAME fitted bins.
X_test_t = binned.transform(X_test,metric='woe')
X_test_t.head()

# Phase 5: Modelling (Train Data )

_logistic regression on WoE features ._

In [0]:
# VIF (multicollinearity) on the WoE features.
X_with_constant = sm.add_constant(X_test_t)

#Creata DF to hold VIF results
vif_data = pd.DataFrame()
vif_data['Features'] = X_with_constant.columns

#calculate VIF for each feature
vif_data['VIF'] = [variance_inflation_factor(X_with_constant,i)
                   for i in range(X_with_constant.shape[1])]

In [0]:
# Values near 1 mean the WoE features are near-orthogonal
# (expected after dropping the correlated raw variables).
vif_data

## Logistic Regression Fit 

In [0]:
X_train_t

In [0]:
# CHAMPION MODEL: unregularised logistic regression via GLM(Binomial).
X_train_t_tranf = sm.add_constant(X_train_t)
model = sm.GLM(Y_train,X_train_t_tranf,family = sm.families.Binomial()).fit()
model.summary()

## Final Feature Lock 

#Phase 6: Validation - Train , Test 

_Discrimination (AUC/Gini/KS) reported on both train and test, with 1000-sample bootstrap 95% confidence intervals to demonstrate the metrics are stable rather than split-specific._

## GINI/AUC 

In [0]:
# Train ROC curve (TPR vs FPR across thresholds).
#Creating ROC
pred_train = model.predict(X_train_t_tranf)
fpr , tpr , threshold = sk.metrics.roc_curve(Y_train,pred_train)
plt.plot(fpr,tpr)

In [0]:
# Train AUC and Gini (Gini = 2*AUC - 1): rank-ordering / discrimination power on train.
auc = sk.metrics.roc_auc_score(Y_train,pred_train)
print(f'AUC:{auc}')
gini = 2*auc - 1
print(f'GINI:{gini}') # Measure how much better is the model at seperating good from bad 

In [0]:
# Score TEST with the locked model.
X_test_t_tranf = sm.add_constant(X_test_t)

pred_test = model.predict(X_test_t_tranf)

In [0]:
# Test ROC curve.
fpr , tpr , threshold = sk.metrics.roc_curve(Y_test,pred_test)
plt.plot(fpr,tpr)

In [0]:
# Test AUC / Gini: the headline OUT-OF-SAMPLE discrimination.
# A small train->test drop indicates no meaningful overfit.
auc_test = sk.metrics.roc_auc_score(Y_test,pred_test)
print(f'AUC of test:{auc_test}')
gini = 2*auc_test - 1
print(f'GINI : {gini}')

In [0]:
# Bootstrap: resample TEST 1000x (with replacement) to get a 95% CI on AUC/Gini.
# Shows the metric is stable, not an artefact of one lucky split.
bs_auc_list = []

for i in range(0,1001):
    random_gen = np.random.choice(X_test_t_tranf.index,size = len(X_test_t_tranf),replace=True)
    random_gen,len(random_gen),len(X_test_t_tranf)
    bs_x = X_test_t_tranf.loc[random_gen]
    bs_y = Y_test.loc[random_gen]
    bs_pred = model.predict(bs_x)
    bs_auc = sk.metrics.roc_auc_score(bs_y,bs_pred)
    bs_auc_list.append(bs_auc)
print(len(bs_auc_list))
print(f'95% CI AUC : {np.percentile(bs_auc_list,2.5) , np.percentile(bs_auc_list,97.5)}')
print(f'95% CI GINI : {2*(np.percentile(bs_auc_list,2.5))-1 , 2*(np.percentile(bs_auc_list,97.5))-1}')

## KS

### Train

In [0]:
# Rank predictions into deciles (1 = riskiest), join the true label.
pred_train_df = pred_train.to_frame(name='Prob')
pred_train_df.sort_values(by='Prob',ascending=False,inplace=True)
pred_train_df['Decile'] = pd.qcut(pred_train_df['Prob'],10,labels=False) + 1
len(pred_train_df)
pred_train_df = pred_train_df.join(Y_train,how='inner')
len(pred_train_df)

In [0]:
# KS table: cumulative good% vs cumulative bad% by decile.
# KS = max vertical gap between the two cumulative curves (peak separation power).
ks = pred_train_df.groupby('Decile').agg(Total=('loan_status','count'), Bad=('loan_status','sum'))
ks['Good'] = ks['Total'] - ks['Bad']
ks['Cum_Good'] = ks['Good'].cumsum()
ks['Cum_Bad'] = ks['Bad'].cumsum()

ks['Cum_Good_perc'] = ks['Cum_Good']/max(ks['Cum_Good'])
ks['Cum_bad_perc'] = ks['Cum_Bad']/max(ks['Cum_Bad'])
ks['KS'] = np.absolute(ks['Cum_bad_perc']-ks['Cum_Good_perc'])

In [0]:
# Report the max KS and the decile at which it occurs.
print(ks)
print(f'Max KS {np.max(ks['KS'])} at Decile {np.argmax(ks['KS'])+1}')

### Test

In [0]:
# Same KS construction on TEST.
pred_test_df = pred_test.to_frame(name='Prob')
pred_test_df.sort_values(by='Prob',ascending=False,inplace=True)
pred_test_df['Decile'] = pd.qcut(pred_test_df['Prob'],10,labels=False) + 1
len(pred_test_df)
pred_test_df = pred_test_df.join(Y_test,how='inner')
len(pred_test_df)

ks_test = pred_test_df.groupby('Decile').agg(Total=('loan_status','count'), Bad=('loan_status','sum'))
ks_test['Good'] = ks_test['Total'] - ks_test['Bad']
ks_test['Cum_Good'] = ks_test['Good'].cumsum()
ks_test['Cum_Bad'] = ks_test['Bad'].cumsum()

ks_test['Cum_Good_perc'] = ks_test['Cum_Good']/max(ks_test['Cum_Good'])
ks_test['Cum_bad_perc'] = ks_test['Cum_Bad']/max(ks_test['Cum_Bad'])
ks_test['KS'] = np.absolute(ks_test['Cum_bad_perc']-ks_test['Cum_Good_perc'])

print(ks_test)
print(f'Max KS {np.max(ks_test['KS'])} at Decile {np.argmax(ks_test['KS'])+1}')

### Bootstrap KS

In [0]:
# Bootstrap KS on TEST for a 95% CI -> stability of the separation metric.
ks_test_list = []


for i in range(0,1001):
    random_gen = np.random.choice(X_test_t_tranf.index,size = len(X_test_t_tranf),replace=True)
    random_gen,len(random_gen),len(X_test_t_tranf)
    bs_x = X_test_t_tranf.loc[random_gen]
    bs_y = Y_test.loc[random_gen]
    bs_pred_ks = model.predict(bs_x)

    pred_test_df = bs_pred_ks.to_frame(name='Prob')
    pred_test_df.sort_values(by='Prob',ascending=False,inplace=True)
    pred_test_df['Decile'] = pd.qcut(pred_test_df['Prob'],10,labels=False) + 1
    len(pred_test_df)
    pred_test_df = pred_test_df.join(bs_y,how='inner')
    len(pred_test_df)

    ks_test = pred_test_df.groupby('Decile').agg(Total=('loan_status','count'), Bad=('loan_status','sum'))
    ks_test['Good'] = ks_test['Total'] - ks_test['Bad']
    ks_test['Cum_Good'] = ks_test['Good'].cumsum()
    ks_test['Cum_Bad'] = ks_test['Bad'].cumsum()

    ks_test['Cum_Good_perc'] = ks_test['Cum_Good']/max(ks_test['Cum_Good'])
    ks_test['Cum_bad_perc'] = ks_test['Cum_Bad']/max(ks_test['Cum_Bad'])
    ks_test['KS'] = np.absolute(ks_test['Cum_bad_perc']-ks_test['Cum_Good_perc'])
    ks_test_list.append(np.max(ks_test['KS']))
print(f'The KS CI 95% : {np.percentile(ks_test_list,2.5)} - {np.percentile(ks_test_list,97.5)}')

# Phase 7: Calibration & Scaling 

_Turning probabilities into an interpretable points scale (odds double every 20 points, anchored at 600 = 50:1)._

##Score Scaling (PDO Method)

In [0]:
# PDO score scaling (Siddiqi). Score = offset + factor*ln(odds).
#   PDO = 20      -> the odds DOUBLE every 20 points
#   base = 600 @ 50:1 odds (base_odds) -> the anchor point
#   factor = PDO/ln(2),  offset = base - factor*ln(base_odds)
# Per-attribute points spread the intercept and offset evenly across the n characteristics.
#Formula : Score = offset + factor*ln(odds)
base = 600
pdo = 20
factor = 20/np.log(2)
offset = 600 - (20/np.log(2))*np.log(50)
#Intercept
beta_z = model.params[0]# Beta 0 
#Sanity check
print(offset + factor*np.log(50) == 600)

#Points per bin = 
#np.sum(-( woe*b1 + b0/N )*factor + offset/N)

params = model.params[1:].tolist()
param_df = pd.DataFrame({'Coeff':params},index = model.params[1:].index)


# Creating Scorecard
scorecard = pd.DataFrame()
# Creating a scorecard df mapping woe to feature and bins 
for j in param_df.index:
    binning = binned.get_binned_variable(j)
    df = pd.DataFrame({'Feature':j,'Bin':binning.binning_table.build()['Bin'].tolist()[0:-1],'WoE':binning.binning_table.build()['WoE'].tolist()[0:-1],'Coef':param_df.loc[j]['Coeff']})
    scorecard = pd.concat([scorecard, df], ignore_index=True)

#Calculated Points per woe BIN 
scorecard['Points'] = ( -(scorecard['WoE']*scorecard['Coef']+ (beta_z/scorecard['Feature'].nunique()) )*factor + offset/scorecard['Feature'].nunique())

In [0]:
scorecard

## Calibration

In [0]:
# Preview test WoE frame ahead of the calibration table.
X_test_t.head()

In [0]:
# Inspect raw train predictions (predicted PDs).
pred_train

In [0]:
# Inspect train target.
Y_train

In [0]:
# CALIBRATION (train): bin predicted PDs into deciles, compare mean PREDICTED PD vs the
# OBSERVED bad rate per decile. A WoE logistic regression is inherently calibrated. So this is a simple formality

# for the XGBoost challenger.
pred_train_df = pred_train.to_frame(name='Prob')
pred_train_df['Decile'] = pd.qcut(pred_train_df['Prob'],10,labels=False)+1
pred_train_df_m = pd.merge(left = pred_train_df , right = Y_train , left_index = True , right_index = True)
calib_train  = pred_train_df_m.groupby('Decile').agg(Pred_Default = ('Prob','mean') , Total = ('Prob','count') , Bad = ('loan_status','sum'))
calib_train['Actual_Default'] = calib_train['Bad']/calib_train['Total']
calib_train['Difference%'] = round(calib_train.Pred_Default - calib_train.Actual_Default,3)*100
calib_train

In [0]:
# Global calibration check: mean predicted PD should ~= mean actual default (train).
print(f"Mean Predicted default: {np.mean(pred_train)}")
print(f"Mean Actual Default: {np.mean(Y_train)}")

In [0]:
# Same decile calibration table on TEST.
pred_test_df = pred_test.to_frame(name='Prob')
pred_test_df['Decile'] = pd.qcut(pred_test_df['Prob'],10,labels=False)+1
pred_test_df_m = pd.merge(left = pred_test_df , right = Y_test , left_index = True , right_index = True)
calib_test  = pred_test_df_m.groupby('Decile').agg(Pred_Default = ('Prob','mean') , Total = ('Prob','count') , Bad = ('loan_status','sum'))
calib_test['Actual_Default'] = calib_test['Bad']/calib_test['Total']
calib_test['Difference%'] = round(calib_test.Pred_Default - calib_test.Actual_Default,3)*100
calib_test

In [0]:
# Calibration plot vs the 45-degree line (points on the line = well calibrated).
plt.plot(calib_test['Actual_Default'],calib_test['Pred_Default'])
plt.axline([0,0],[1,1])
plt.ylabel('Pred')
plt.xlabel('Actual')

In [0]:
# Global calibration check on TEST.
print(f"Mean Predicted default: {np.mean(pred_test)}")
print(f"Mean Actual Default: {np.mean(Y_test)}")

# Final Model Artifact Lock 


_Freezing binning object, model, points table and metrics, logged to MLflow with a champion tag_

In [0]:
# Re-apply the locked median imputation to loan_int_rate on train
# (idempotent safety step right before the artifact lock).
X_train['loan_int_rate'] = X_train['loan_int_rate'].fillna(median_value)
X_train['loan_int_rate'].isna().sum() == 0

In [0]:
# Final binning summary (locked).
binned.summary()

In [0]:
# Final locked feature list + model coefficients/intercept (the model spec).
# Final Feature 
print(X_train_t.columns)
# Log regression coeff Logistic regression coefficients + intercept (from statsmodels)
model.params

In [0]:
# PDO scaling parameters (base score, base odds, PDO, factor, offset)
#Formula : Score = offset + factor*ln(odds)
# base = 600
# pdo = 20
# base_odds = 50
# factor = 20/np.log(2)
# offset = 600 - (20/np.log(2))*np.log(50)
# #Intercept
# beta_z = -1.3613
# # Points table (attribute → points mapping)
# scorecard

In [0]:
# Train: Gini, KS, AUC


# AUC: 0.87
# GINI: 0.75
# KS:  0.59
# Test: Gini, KS, AUC

# Test
# AUC: 0.86
# GINI: 0.73
# KS: 0.58
# Calibration table (decile-level predicted vs observed) (Test Data)
# Mean Predicted default: 0.220
# Mean Actual Default: 0.218

In [0]:
#Uploaded artifacts to volume
import mlflow
mlflow.disable_system_metrics_logging()

VOL = "/Volumes/workspace/default/model_artifacts/kaggle_log"
import os; os.makedirs(VOL, exist_ok=True)

joblib.dump(binned, f"{VOL}/woe_binning.pkl")
joblib.dump(model,  f"{VOL}/lr_model.pkl")
scorecard.to_csv(f"{VOL}/points_table.csv", index=False)

with mlflow.start_run(run_name="scorecard_artifact_lock"):
    mlflow.set_tag("stage", "champion")
    mlflow.log_param("artifact_volume", VOL) 